## Camada Gold Produtiva
Iremos adaptar o nosso código 3_3_ingestao_gold.ipynb

Obs: precisamos fazer a ingestão da tabela de clientes no catálogo prod.


In [0]:
# criação do Schema Gold

spark.sql("""
    CREATE DATABASE IF NOT EXISTS workspace.gold
    COMMENT 'Schema da Camada Gold'
""")

print("✓ Schema 'workspace.gold' criado/verificado com sucesso!")
# Criação da tabela Gold: atendimentos enriquecidos com informações de clientes
# Tabela otimizada para análises e dashboards

spark.sql("""
    CREATE OR REPLACE TABLE workspace.gold.gold_atendimentos_enriquecidos
    USING DELTA
    PARTITIONED BY (data_particao)
    COMMENT 'Tabela gold com atendimentos enriquecidos com informações de clientes para análises e dashboards'
    AS
    SELECT 
        -- Identificadores
        a.id_interacao,
        a.cliente_id,
        
        -- Informações do atendimento
        upper(a.canal) AS canal,
        upper(a.status) AS status,
        upper(a.departamento) AS departamento,
        a.data_hora,
        a.data_particao,
        
        -- Informações do cliente (LEFT JOIN)
        c.idade AS cliente_idade,
        c.estado_residencia AS cliente_estado,
        c.qtd_produtos_adquiridos AS cliente_qtd_produtos,
        upper(c.categoria_fidelidade) AS cliente_categoria_fidelidade,
        
        -- Métricas de WhatsApp
        a.whatsapp_numero_origem,
        SUBSTRING(a.whatsapp_numero_origem, 4, 2) AS whatsapp_ddd,
        CASE WHEN a.whatsapp_atendente_bot='true' THEN 'BOT' else 'HUMANO' end as whatsapp_atendente,
        a.whatsapp_tempo_resposta_bot_seg,
        
        -- Métricas de Telefone
        a.telefone_duracao_chamada_seg,
        a.telefone_fila_espera_seg,
        a.telefone_protocolo,
        a.telefone_transferencias,
        
        -- Métricas de Email
        a.email_dominio,
        a.email_tamanho_corpo_bytes,
        a.email_anexos_quantidade,
        a.email_tempo_primeira_resposta_horas,
        
        -- Métricas de Chat
        a.chat_browser,
        a.chat_pagina_origem,
        a.chat_satisfacao_pre_atendimento,
        
        -- Metadados de processamento
        a.silver_processing_timestamp,
        CURRENT_TIMESTAMP() AS gold_processing_timestamp
        
    FROM workspace.silver.silver_atendimentos a
    LEFT JOIN workspace.raw.tb_info_clientes c
        ON a.cliente_id = c.cliente_id
""")

print("✅ Tabela gold_atendimentos_enriquecidos criada com sucesso!")